# Install requirements and dependencies

In [ ]:
PROJECT_PATH = "/home/jupyter-gabriel/projects/pricing_automation/pricing-automation/etl_pipeline"

In [ ]:
%pip install -r "$PROJECT_PATH/requirements.txt"

## Load kedro with magic command

This will create 3 objects available in the enviroment:
1. session (a kedro.framework.session object which is capable of creating new sessions, running pipelines or nodes)
2. context (a kedro.framework.context object which contains, among other specification, the resolve catalog)
3. catalog (an object capable of loading and saving the catalog entries defined in the catalog.yml)


In [ ]:
%load_ext kedro.ipython

Change the notebook to the project path and reload the project from there

In [ ]:
%cd $PROJECT_PATH
%reload_kedro .

## (Optional) Load custom functions
If you want to manually test the functions you build load them

You can manually run each step and take advantage of the Kedro Catalog Utility to load and save datasets specified in the catalog

OR, you can get rid of Kedro entirely by making manual loadings and savings knowing that each function requires its own inputs and outputs

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pricing_automation.pipelines.utils import *
from pricing_automation.pipelines.a01_aoi_period import *
from pricing_automation.pipelines.n01_extract_data import *
from pricing_automation.pipelines.n02_process_data import *
from pricing_automation.pipelines.n03_create_triggers import *

# Create a Session

This lets you set quick parameters to overrride the ones in the yml files

### Step 0: Define parameters to override at runtime

In [ ]:
%reload_kedro
runtime_params = {
    'country': 'colombia', # Local folder name to store data 
    'region': 'valles',
    'lead_id': 'hevelma_seguros', # Lead name
    'provider': 'UCSB', # Data provider, can be 'ERA5' or 'UCSB'
    'field': 'prcp', # Short variable name, can be 'swc', 'prcp', 'tmin', 'tmax'
    'start_year': 2006, # (optional) Use this to override the initial year of data
    'end_year': 2025, # (optional) Use this to override the ending year of data
}
session_trial = session.create(
    runtime_params = runtime_params
)
# (Optional) Load the catalog with the context provided earlier for quick loads
catalog_trial = session_trial.load_context().catalog 

Now you can call and load any entry in the Catalog, provided it exists, by running:

__catalog_trial.load('namespace.entry')__

### Step 1. Get Area Of Interest

It either creates a bounding box area with the parameters in __params.yml__ (provided the _override_ parameter is set to __true__

Otherwise it will attempt to create a bounding box from the _gdf_request_ entry in the __catalog.yml__)

You can use a large geometry and subset it on the fly. To do this use the _include_ and _exclude_ parameter in _params_s_ in the  __params.yml__ file. You should provide a dictionary of the form 

* 'column': ['value1', 'value2']

E.g.: For Argentina the Province of Buenos Aires can be filtered with 'depto' = '06' so use:

* 'include': {'depto': ['06']}

The same logic applies if it is easier to exclude some values. E.g.: Excluding the Province of Buenos Aires

* 'exclude': {'depto': ['06']}

### Step 2. Extract data

In [ ]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    tags=['extract']
)

### Step 3. Process data

In [ ]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    tags=['process']
    #node_names = ['swc.get_aoi']
)

### Step 4. Create triggers

In [ ]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    tags=['triggers']
    #node_names = ['swc.get_aoi']
)

### Step 5. Bootstrap aep

In [ ]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    tags=['aep']
)

### Step 7. Compute pricing quote

In [ ]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    tags=['pricing']
)

# Step 8. Create auxiliary plots

In [ ]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    tags=['viz']
)

# Complete Run

There is no need to run each step part by part, Kedro handles dependencies at runtime.

So you can specify a list of tags to run or just run the whole pipeline from step 1 to step 7.

The _extract_data_ takes the longest to complete (about an hour for 30 years of data if AOI is ~ 4 by 4 degrees in size).

The rest of the steps are relatively faster (under 3 minutes each and some take seconds)

In [ ]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    #tags=['extract', 'process', 'triggers', 'pricing']
)